# Notebook 2 · Hierarchical Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Toy Use Case

---

## What is a Hierarchical Architecture?

In a **hierarchical MAS** one agent (the *manager*) controls the workflow.
The manager plans, delegates sub-tasks to *worker* agents, and then synthesizes
their outputs into the final result.
Workers never talk to each other directly — all communication flows through the manager.

```
                  ┌──────────────┐
                  │   MANAGER    │  ← plans, delegates, synthesizes
                  └──┬───┬───┬───┘
         ┌───────────┘   │   └───────────┐
    ┌────▼────┐     ┌────▼────┐     ┌────▼────┐
    │Dest     │     │Booking  │     │Activity │
    │Worker   │     │Worker   │     │Worker   │
    └─────────┘     └─────────┘     └─────────┘
          worker → manager, never worker → worker
```

### Key Properties
| Property | Hierarchical |
|---|---|
| Authority | Concentrated in the manager |
| Communication | Worker → manager (star topology) |
| Stop condition | Manager decides when work is done |
| Failure mode | Single point of failure at the manager |

### When to Use It
Hierarchical designs suit tasks with a clear decomposition into subtasks,
where you want a single agent to enforce quality or consistency, and where
parallel or contradictory worker output needs to be reconciled.

---

## What We Will Build

One **manager** produces a step-by-step plan, then dispatches three **workers**:

- **Destination worker** — picks the destination and travel period
- **Booking worker** — picks one flight and one hotel
- **Activity worker** — picks activities to match the request

The manager collects all worker outputs and writes the final itinerary.
Workers never see each other's outputs directly.


## 1 · Setup: One Model, One Request, One Catalog

The same three blocks appear in every notebook of this series:

| Block | Purpose |
|---|---|
| `model` | One small, cheap LLM used by every agent |
| `USER_REQUEST` | The single traveler request we want to fulfill |
| `CATALOG` + helpers | The only data source agents are allowed to use |

Keeping these identical lets you compare orchestration patterns in isolation.
Nothing changes between notebooks except *how agents talk to each other*.

> **Prerequisite:** set `OPENAI_API_KEY` in your environment before running.


In [1]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# ── One model powers every agent in the series ──────────────────────────────
# We use gpt-4.1-mini for speed and low cost during learning.
# Swap to any model that supports structured output (temperature=0 keeps
# outputs deterministic so you can rerun and compare results).
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# ── The traveler request ─────────────────────────────────────────────────────
# All four notebooks solve this exact request.
# The only thing that changes is the orchestration architecture.
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use destinations, flights, hotels, and activities
# listed below. This constraint keeps the toy example clean and comparable.

DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",         "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",         "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",          "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",          "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",      "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel",   "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
# Simple filter functions so agents can look up options by destination.

def flights_for(destination: str) -> list[dict]:
    return [f for f in FLIGHTS if f["destination"] == destination]

def hotels_for(destination: str) -> list[dict]:
    return [h for h in HOTELS if h["destination"] == destination]

def activities_for(destination: str) -> list[dict]:
    return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    """Return the full catalog as a plain-text string suitable for a prompt."""
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Define the Manager Planning Schema

The manager's job is to break the travel request into an ordered list of tasks.
Each task specifies which worker should run and what exactly that worker must do.

We use `with_structured_output` so the manager returns a `ManagerPlan` object
that the orchestration loop can iterate over directly — no string parsing needed.

### Why structured output for the manager?
Structured output eliminates the need to parse the manager's instructions.
The orchestration code becomes a clean `for task in plan.tasks` loop,
which is the most beginner-readable way to show top-down control.


In [2]:
from typing import Literal

# ── Manager output schema ──────────────────────────────────────────────────────
# WorkerTask names the worker and gives it a concrete instruction.
# ManagerPlan is just an ordered list of WorkerTask objects.

class WorkerTask(BaseModel):
    worker:      Literal["destination_worker", "booking_worker", "activity_worker"]
    instruction: str = Field(description="A precise task written for that worker.")

class ManagerPlan(BaseModel):
    tasks: list[WorkerTask] = Field(description="Ordered list of tasks for the workers.")

# Bind the schema so manager_llm.invoke() returns a ManagerPlan directly.
manager_llm = model.with_structured_output(ManagerPlan)

# ── Worker registry ────────────────────────────────────────────────────────────
# Each entry describes the worker's fixed responsibility.
# Workers never see this dict — it is only for the orchestration layer.
WORKERS = {
    "destination_worker": "Choose the best destination and travel period for the request.",
    "booking_worker":     "Choose one flight and one hotel that fit the destination and budget.",
    "activity_worker":    "Choose activities that match the traveler's interests.",
}

# ── Worker runner ──────────────────────────────────────────────────────────────
def run_worker(worker_name: str, instruction: str, manager_notes: str) -> str:
    """
    Call one worker agent.

    Key hierarchical properties visible here:
    - The worker receives a direct instruction from the manager.
    - The worker sees manager_notes (the manager's running log) but
      NOT the other workers' raw outputs — the manager summarises those.
    - The worker returns a plain string back to the manager.
    """
    messages = [
        SystemMessage(content=(
            "You are a worker agent in a hierarchical travel-agency MAS.\n"
            "You take instructions from the manager and report back to the manager.\n"
            "Do not communicate with other workers. "
            "Only use options from the catalog.\n\n"
            f"Your fixed responsibility: {WORKERS[worker_name]}"
        )),
        HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Manager notes so far:\n{manager_notes or 'No notes yet.'}\n\n"
            f"Your assigned task:\n{instruction}"
        )),
    ]
    return model.invoke(messages).content

print("Manager schema and workers defined.")
print("Workers:", list(WORKERS.keys()))


Manager schema and workers defined.
Workers: ['destination_worker', 'booking_worker', 'activity_worker']


## 3 · Execute the Hierarchy

The orchestration follows a classic **plan → delegate → synthesize** pattern.

### Step-by-step
1. **Manager plans** — produces an ordered `ManagerPlan` from the user request.
2. **Workers execute** — each worker runs in the order the manager specified.
3. **Manager synthesizes** — collects all worker outputs and writes the final answer.

> Notice how the orchestration code is almost entirely about *routing*:
> sending the manager's instructions to the right worker and collecting results.
> That is the essence of a hierarchical architecture.


In [3]:
# ── Step 1: Manager creates the plan ─────────────────────────────────────────
# The manager reads the user request and the list of available workers,
# then returns a structured plan (an ordered list of WorkerTask objects).
plan = manager_llm.invoke([
    SystemMessage(content=(
        "You are the manager in a hierarchical travel-agency MAS.\n"
        "Break the traveler's request into an ordered list of tasks "
        "using only the workers listed below. Keep the plan concise."
    )),
    HumanMessage(content=(
        f"Traveler request:\n{USER_REQUEST}\n\n"
        f"Available workers and their responsibilities:\n"
        + "\n".join(f"  {k}: {v}" for k, v in WORKERS.items())
    )),
])

print("Manager plan:")
for i, task in enumerate(plan.tasks, 1):
    print(f"  {i}. {task.worker}: {task.instruction}")

# ── Step 2: Workers execute in the manager's order ───────────────────────────
manager_notes = []  # the manager's running log — grows after each worker call

for task in plan.tasks:
    # Pass the accumulated notes so each worker has context from earlier steps.
    notes_text = "\n".join(manager_notes)
    result = run_worker(task.worker, task.instruction, notes_text)

    # The manager records each result. This is the star-topology in action:
    # information flows worker → manager, never worker → worker.
    manager_notes.append(f"{task.worker}:\n{result}")

    print(f"\n[{task.worker}] done — {len(result)} characters")

# ── Step 3: Manager synthesizes the final itinerary ──────────────────────────
# The same model instance acts as manager again.
# This reflects the real-world pattern where the manager both delegates
# and integrates the final answer.
worker_output_text = "\n\n".join(manager_notes)

final_itinerary = model.invoke([
    SystemMessage(content=(
        "You are the same manager. "
        "Combine the worker outputs into one clean, client-ready travel itinerary."
    )),
    HumanMessage(content=(
        f"Traveler request:\n{USER_REQUEST}\n\n"
        f"Worker outputs:\n{worker_output_text}"
    )),
]).content

# ── Results ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("MANAGER PLAN")
print("="*60)
for i, task in enumerate(plan.tasks, 1):
    print(f"  {i}. → {task.worker}")
    print(f"       {task.instruction}")

print("\n" + "="*60)
print("WORKER OUTPUTS (as seen by the manager)")
print("="*60)
for note in manager_notes:
    print("\n" + note)

print("\n" + "="*60)
print("FINAL ITINERARY (manager synthesis)")
print("="*60)
print(final_itinerary)


Manager plan:
  1. destination_worker: Select the best destination and travel period for a 4-day spring trip from Rome with a mid-range budget and easy flights.
  2. booking_worker: Book one easy flight and one centrally located hotel within the mid-range budget for the chosen destination and travel period.
  3. activity_worker: Plan a simple daily itinerary mixing food and cultural activities for the 4-day trip at the chosen destination.

[destination_worker] done — 639 characters

[booking_worker] done — 346 characters

[activity_worker] done — 753 characters

MANAGER PLAN
  1. → destination_worker
       Select the best destination and travel period for a 4-day spring trip from Rome with a mid-range budget and easy flights.
  2. → booking_worker
       Book one easy flight and one centrally located hotel within the mid-range budget for the chosen destination and travel period.
  3. → activity_worker
       Plan a simple daily itinerary mixing food and cultural activities for the 4-d

## 4 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| Manager produces a **structured plan** | Clean routing — no string parsing needed |
| Workers receive **direct instructions** | Workers are focused; they cannot deviate |
| Workers **never talk to each other** | The star topology keeps communication traceable |
| Manager **synthesizes** the final answer | A single agent is responsible for quality |

### How this differs from the other notebooks
- **vs Flat (nb 1):** a manager controls the order; peers cannot contradict each other
- **vs Society (nb 3):** one agent decides, rather than the group voting
- **vs Team (nb 4):** workers are directed per-task; team members own standing sections

### When the hierarchical pattern struggles
- **Manager bottleneck** — every output passes through one agent, so its quality
  constrains the whole system
- **Sequential execution** — workers run in order, which is slower than parallel
- **Brittle plans** — if the manager misassigns a task, no worker can self-correct
